# Notebook 02 — Tests statistiques
**Projet :** Prévision du crédit bancaire dans la zone UEMOA (2000-2024)  
**Phase J3 :** Stationnarité, cointégration, Hausman, hétéroscédasticité, autocorrélation

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from pathlib import Path
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

DATA_PATH = Path('../data/panel_uemoa_complet.csv')
df = pd.read_csv(DATA_PATH)
df = df.sort_values(['iso3', 'annee']).reset_index(drop=True)
print(f"Panel chargé : {df.shape[0]} obs × {df.shape[1]} variables")

Panel chargé : 200 obs × 28 variables


## 1. Tests de stationnarité par série (ADF + KPSS)

- **ADF** : H0 = présence de racine unitaire (non-stationnaire)
- **KPSS** : H0 = série stationnaire
- Un consensus ADF-reject + KPSS-accept → série stationnaire I(0)

In [2]:
VARS_TEST = [
    'credit_prive_pib', 'pib_croissance', 'inflation_ipc',
    'masse_monetaire_m2_pib', 'imf_dette_publique',
    'ouverture_commerciale', 'ide_pib', 'urbanisation',
    'wgi_controle_corruption', 'bceao_taux_directeur', 'imf_balance_courante'
]

def adf_test(serie, maxlag=2):
    s = serie.dropna()
    if len(s) < 8:
        return np.nan, np.nan, 'N/A'
    result = adfuller(s, maxlag=maxlag, autolag='AIC')
    stat, p = result[0], result[1]
    conclusion = 'Stationnaire I(0)' if p < 0.05 else 'Non-stationnaire'
    return stat, p, conclusion

def kpss_test(serie):
    s = serie.dropna()
    if len(s) < 8:
        return np.nan, np.nan, 'N/A'
    result = kpss(s, regression='c', nlags='auto')
    stat, p = result[0], result[1]
    conclusion = 'Stationnaire' if p > 0.05 else 'Non-stationnaire'
    return stat, p, conclusion

resultats = []
for pays in sorted(df['iso3'].unique()):
    for var in VARS_TEST:
        serie = df[df['iso3'] == pays][var]
        adf_stat, adf_p, adf_conc = adf_test(serie)
        kpss_stat, kpss_p, kpss_conc = kpss_test(serie)
        accord = 'OK' if (adf_conc == 'Stationnaire I(0)' and kpss_conc == 'Stationnaire') or \
                         (adf_conc == 'Non-stationnaire' and kpss_conc == 'Non-stationnaire') else 'Divergent'
        resultats.append({
            'Pays': pays, 'Variable': var,
            'ADF_stat': adf_stat, 'ADF_p': adf_p, 'ADF': adf_conc,
            'KPSS_stat': kpss_stat, 'KPSS_p': kpss_p, 'KPSS': kpss_conc,
            'Accord': accord
        })

res_df = pd.DataFrame(resultats)
print("=== Résumé par variable (proportion stationnaire selon ADF) ===")
summary = res_df.groupby('Variable')['ADF'].apply(lambda x: f"{(x=='Stationnaire I(0)').sum()}/{len(x)} pays")
print(summary.to_string())

=== Résumé par variable (proportion stationnaire selon ADF) ===
Variable
bceao_taux_directeur       0/8 pays
credit_prive_pib           0/8 pays
ide_pib                    4/8 pays
imf_balance_courante       3/8 pays
imf_dette_publique         1/8 pays
inflation_ipc              7/8 pays
masse_monetaire_m2_pib     0/8 pays
ouverture_commerciale      1/8 pays
pib_croissance             8/8 pays
urbanisation               1/8 pays
wgi_controle_corruption    0/8 pays


In [3]:
print("\n=== Détail pour la variable CIBLE : credit_prive_pib ===")
cible_res = res_df[res_df['Variable'] == 'credit_prive_pib'][['Pays','ADF_p','ADF','KPSS_p','KPSS','Accord']]
print(cible_res.to_string(index=False))


=== Détail pour la variable CIBLE : credit_prive_pib ===
Pays  ADF_p              ADF  KPSS_p             KPSS Accord
 BEN 0.5970 Non-stationnaire  0.0200 Non-stationnaire     OK
 BFA 0.7938 Non-stationnaire  0.0140 Non-stationnaire     OK
 CIV 0.9944 Non-stationnaire  0.0142 Non-stationnaire     OK
 GNB 0.7638 Non-stationnaire  0.0212 Non-stationnaire     OK
 MLI 0.8252 Non-stationnaire  0.0167 Non-stationnaire     OK
 NER 0.3974 Non-stationnaire  0.0183 Non-stationnaire     OK
 SEN 0.7223 Non-stationnaire  0.0129 Non-stationnaire     OK
 TGO 0.7617 Non-stationnaire  0.0185 Non-stationnaire     OK


## 2. Tests en différence première

Pour les séries non-stationnaires, on teste si elles le deviennent en première différence → I(1)

In [4]:
vars_non_stat = res_df[res_df['ADF'] == 'Non-stationnaire']['Variable'].unique()
print(f"Variables non-stationnaires (ADF niveau) : {list(vars_non_stat)}")

if len(vars_non_stat) > 0:
    resultats_diff = []
    for pays in sorted(df['iso3'].unique()):
        for var in vars_non_stat:
            serie = df[df['iso3'] == pays][var].diff().dropna()
            adf_stat, adf_p, adf_conc = adf_test(serie)
            resultats_diff.append({'Pays': pays, 'Variable': f'Δ{var}', 'ADF_p': adf_p, 'ADF': adf_conc})
    
    res_diff = pd.DataFrame(resultats_diff)
    print("\n=== Résumé — tests ADF en 1ère différence ===")
    summary_diff = res_diff.groupby('Variable')['ADF'].apply(lambda x: f"{(x=='Stationnaire I(0)').sum()}/{len(x)} pays")
    print(summary_diff.to_string())
else:
    print("Toutes les séries sont stationnaires en niveau.")

Variables non-stationnaires (ADF niveau) : ['credit_prive_pib', 'masse_monetaire_m2_pib', 'imf_dette_publique', 'ouverture_commerciale', 'ide_pib', 'urbanisation', 'wgi_controle_corruption', 'bceao_taux_directeur', 'imf_balance_courante', 'inflation_ipc']

=== Résumé — tests ADF en 1ère différence ===
Variable
Δbceao_taux_directeur       8/8 pays
Δcredit_prive_pib           8/8 pays
Δide_pib                    6/8 pays
Δimf_balance_courante       8/8 pays
Δimf_dette_publique         7/8 pays
Δinflation_ipc              8/8 pays
Δmasse_monetaire_m2_pib     8/8 pays
Δouverture_commerciale      8/8 pays
Δurbanisation               2/8 pays
Δwgi_controle_corruption    6/8 pays


## 3. Tests de racine unitaire en panel (IPS)

Le test **Im-Pesaran-Shin (IPS)** est adapté aux panels hétérogènes (T > N).  
H0 = toutes les séries ont une racine unitaire

In [5]:
try:
    from linearmodels.panel.unitroot import IPS
    
    df_panel = df.set_index(['iso3', 'annee'])
    print("=== Test IPS (Im-Pesaran-Shin) sur le panel complet ===")
    
    for var in ['credit_prive_pib', 'masse_monetaire_m2_pib', 'imf_dette_publique', 'urbanisation']:
        serie = df_panel[var].dropna()
        try:
            test = IPS(serie, lags=1)
            res = test.summary
            stat = test.stat
            pval = test.pvalue
            print(f"  {var:<35} stat={stat:.3f}  p={pval:.4f}  → {'I(0)' if pval < 0.05 else 'Racine unitaire possible'}")
        except Exception as e:
            print(f"  {var:<35} Erreur : {e}")
except ImportError:
    print("linearmodels non disponible — test IPS ignoré.")
    print("Alternative : utiliser les résultats ADF individuels (Section 1).")

linearmodels non disponible — test IPS ignoré.
Alternative : utiliser les résultats ADF individuels (Section 1).


## 4. Test de Hausman — Effets fixes vs Effets aléatoires

In [6]:
try:
    from linearmodels.panel import PanelOLS, RandomEffects
    
    VARS_X = [
        'pib_croissance', 'inflation_ipc', 'masse_monetaire_m2_pib',
        'imf_dette_publique', 'ouverture_commerciale', 'ide_pib',
        'wgi_controle_corruption', 'bceao_taux_directeur'
    ]
    
    df_h = df[['iso3', 'annee', 'credit_prive_pib'] + VARS_X].dropna()
    df_h = df_h.set_index(['iso3', 'annee'])
    
    y = df_h['credit_prive_pib']
    X = add_constant(df_h[VARS_X])
    
    fe = PanelOLS(y, df_h[VARS_X], entity_effects=True, time_effects=False)
    fe_res = fe.fit(cov_type='clustered', cluster_entity=True)
    
    re = RandomEffects(y, df_h[VARS_X])
    re_res = re.fit()
    
    from linearmodels.panel.results import compare
    
    b_fe = fe_res.params
    b_re = re_res.params
    b_diff = b_fe - b_re.reindex(b_fe.index)
    
    v_fe = fe_res.cov
    v_re = re_res.cov.reindex(index=b_fe.index, columns=b_fe.index)
    v_diff = v_fe - v_re.values
    
    chi2 = float(b_diff.values @ np.linalg.pinv(v_diff) @ b_diff.values)
    from scipy.stats import chi2 as chi2dist
    p_hausman = chi2dist.sf(chi2, df=len(b_diff))
    
    print(f"=== Test de Hausman ===")
    print(f"  Chi² = {chi2:.3f}  |  ddl = {len(b_diff)}  |  p-valeur = {p_hausman:.4f}")
    if p_hausman < 0.05:
        print("  → Rejet H0 : EFFETS FIXES préférés (hétérogénéité corrélée avec les régresseurs)")
    else:
        print("  → Non-rejet H0 : EFFETS ALÉATOIRES envisageables")
except ImportError:
    print("linearmodels non disponible.")
    print("→ Utiliser le test de Hausman manuellement via statsmodels (voir section alternative ci-dessous)")
except Exception as e:
    print(f"Erreur Hausman : {e}")

Erreur Hausman : float division by zero


### Alternative Hausman (si linearmodels absent)

In [10]:
# Hausman via procédure de Mundlak (alternative robuste)
VARS_X = [
    'pib_croissance', 'inflation_ipc', 'masse_monetaire_m2_pib',
    'imf_dette_publique', 'ouverture_commerciale', 'ide_pib',
    'wgi_controle_corruption', 'bceao_taux_directeur'
]

df_m = df[['iso3', 'annee', 'credit_prive_pib'] + VARS_X].dropna()

# Calcul des moyennes pays (between)
means = df_m.groupby('iso3')[VARS_X].mean()
means.columns = [f'{v}_mean' for v in VARS_X]
df_m = df_m.merge(means, on='iso3')

# Régression Mundlak (effets aléatoires + moyennes within)
X_mundlak = VARS_X + [f'{v}_mean' for v in VARS_X]
X_reg = add_constant(df_m[X_mundlak].values)
y_reg = df_m['credit_prive_pib'].values

model = OLS(y_reg, X_reg).fit()

# Test de significativité conjointe des termes de moyennes
n_means = len(VARS_X)
idx_means = list(range(1 + len(VARS_X), 1 + len(VARS_X) + n_means))

# from statsmodels.stats.compat_pandas import nanmean

F_stat = model.f_test(np.eye(model.params.shape[0])[idx_means]).fvalue
F_pval = model.f_test(np.eye(model.params.shape[0])[idx_means]).pvalue

print(f"=== Test de Mundlak (proxy Hausman) ===")
print(f"  F = {float(F_stat):.3f}  |  p-valeur = {float(F_pval):.4f}")
if float(F_pval) < 0.05:
    print("  → Termes de moyenne significatifs : EFFETS FIXES recommandés")
else:
    print("  → Termes de moyenne non-significatifs : effets aléatoires envisageables")

=== Test de Mundlak (proxy Hausman) ===
  F = 25.934  |  p-valeur = 0.0000
  → Termes de moyenne significatifs : EFFETS FIXES recommandés


## 5. Test de Breusch-Pagan — Hétéroscédasticité

In [11]:
VARS_X_BP = [
    'pib_croissance', 'inflation_ipc', 'masse_monetaire_m2_pib',
    'imf_dette_publique', 'ouverture_commerciale', 'ide_pib'
]
df_bp = df[['credit_prive_pib'] + VARS_X_BP].dropna()
X_bp = add_constant(df_bp[VARS_X_BP])
y_bp = df_bp['credit_prive_pib']
model_bp = OLS(y_bp, X_bp).fit()

bp_stat, bp_p, f_stat, f_p = het_breuschpagan(model_bp.resid, model_bp.model.exog)
print(f"=== Test de Breusch-Pagan (hétéroscédasticité) ===")
print(f"  LM stat = {bp_stat:.3f}  |  p-valeur = {bp_p:.4f}")
if bp_p < 0.05:
    print("  → Hétéroscédasticité détectée → utiliser erreurs standard robustes (clustered) dans les modèles panel")
else:
    print("  → Homoscédasticité non-rejetée")

=== Test de Breusch-Pagan (hétéroscédasticité) ===
  LM stat = 29.254  |  p-valeur = 0.0001
  → Hétéroscédasticité détectée → utiliser erreurs standard robustes (clustered) dans les modèles panel


## 6. Test d'autocorrélation (Wooldridge simplifié)

In [12]:
from statsmodels.stats.stattools import durbin_watson

dw_stats = {}
for pays in sorted(df['iso3'].unique()):
    subset = df[df['iso3'] == pays].sort_values('annee')
    vars_ok = [v for v in VARS_X_BP if subset[v].notna().all()]
    if len(vars_ok) >= 3 and subset['credit_prive_pib'].notna().all():
        X_dw = add_constant(subset[vars_ok])
        y_dw = subset['credit_prive_pib']
        m = OLS(y_dw, X_dw).fit()
        dw_stats[pays] = durbin_watson(m.resid)

print("=== Statistique de Durbin-Watson par pays ===")
print("  (valeur ≈ 2 → pas d'autocorrélation, < 1.5 → autocorrélation positive)")
for pays, dw in dw_stats.items():
    flag = ' ← autocorrélation probable' if dw < 1.5 else ''
    print(f"  {pays} : DW = {dw:.3f}{flag}")

=== Statistique de Durbin-Watson par pays ===
  (valeur ≈ 2 → pas d'autocorrélation, < 1.5 → autocorrélation positive)
  BEN : DW = 0.951 ← autocorrélation probable
  BFA : DW = 1.260 ← autocorrélation probable
  CIV : DW = 1.562
  GNB : DW = 1.505
  MLI : DW = 1.201 ← autocorrélation probable
  NER : DW = 1.965
  SEN : DW = 1.315 ← autocorrélation probable
  TGO : DW = 1.016 ← autocorrélation probable


## 7. Synthèse des tests

| Test | Conclusion | Impact sur la modélisation |
|------|-----------|---------------------------|
| Stationnarité (ADF) | À lire ci-dessus | Spécification ARDL (lags) |
| Hausman | À lire ci-dessus | Effets fixes vs aléatoires |
| Breusch-Pagan | À lire ci-dessus | Erreurs standard robustes |
| Durbin-Watson | À lire ci-dessus | Correction AR(1) si nécessaire |

→ **Étape suivante : Notebook 03 — Modèles ARDL (Bénin) + Panel effets fixes (J4)**